## Libraries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

## Dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_point = 22500
N_t = 77
L_left = 0
L_right = 4380
nx = ny = 150

U_total = torch.zeros((N_point * N_t, 1), dtype=torch.float32, device=device)
X_total = torch.zeros((N_point * N_t, 1), dtype=torch.float32, device=device)
Y_total = torch.zeros((N_point * N_t, 1), dtype=torch.float32, device=device)

for i in range(N_t):
    x = np.linspace(L_left, L_right, nx)
    y = np.linspace(L_left, L_right, ny)
    X, Y = np.meshgrid(x, y)
    X_total[i * N_point : (i + 1) * N_point] = torch.tensor(X.reshape(N_point, 1), dtype=torch.float32, device=device)
    Y_total[i * N_point : (i + 1) * N_point] = torch.tensor(Y.reshape(N_point, 1), dtype=torch.float32, device=device)

U = sio.loadmat('U_total_1')
data = list(U.items())[3][1]
U_data = data[0:N_point * N_t, 0]
U_total = torch.tensor(U_data, dtype=torch.float32, device=device).view(-1, 1).requires_grad_(True)
U_xy1 = torch.cat((X_total, Y_total, U_total), dim=1)
sio.savemat('U_xy1.mat', {'U_xy1': U_xy1.cpu().detach().numpy()}) 

## PINN

In [ ]:
class FCN(nn.Module):
    def __init__(self, N_INPUT, N_OUTPUT, N_HIDDEN, N_LAYERS):
        super().__init__()
        activation = nn.Tanh
        self.fcs = nn.Sequential(nn.Linear(N_INPUT, N_HIDDEN), activation())
        self.fch = nn.Sequential(*[nn.Sequential(nn.Linear(N_HIDDEN, N_HIDDEN), activation()) for _ in range(N_LAYERS-1)])
        self.fce = nn.Linear(N_HIDDEN, N_OUTPUT)

    def forward(self, x):
        x = self.fcs(x)
        x = self.fch(x)
        x = self.fce(x)
        return x

torch.manual_seed(123)
pinn = FCN(3, 1, 20, 3).to(device)

dt = 1/3
t_total = torch.zeros((N_point * N_t, 1), dtype=torch.float32, device=device)
for i in range(N_t):
    T = (i + 1) * dt + 0.001
    t_step = (torch.ones((N_point, 1), dtype=torch.float32, device=device) * T).view(-1, 1)
    t_total[i * N_point : (i + 1) * N_point] = t_step
t_total = t_total.requires_grad_(True)

x_data = sio.loadmat('U_xy1')
data = list(x_data.items())[3][1]
x_d = torch.tensor(data[0:N_point * N_t, 0], dtype=torch.float32, device=device).view(-1, 1).requires_grad_(True)
y_d = torch.tensor(data[0:N_point * N_t, 1], dtype=torch.float32, device=device).view(-1, 1).requires_grad_(True)
u_d = torch.tensor(data[0:N_point * N_t, 2], dtype=torch.float32, device=device).view(-1, 1).requires_grad_(True)

def normalize(data, data_min, data_max):
    return (data - data_min) / (data_max - data_min)

def denormalize(data, data_min, data_max):
    return data * (data_max - data_min) + data_min

x_d_min, x_d_max = torch.min(x_d), torch.max(x_d)
y_d_min, y_d_max = torch.min(y_d), torch.max(y_d)
u_d_min, u_d_max = torch.min(u_d), torch.max(u_d)

x_d_normalized = normalize(x_d, x_d_min, x_d_max)
y_d_normalized = normalize(y_d, y_d_min, y_d_max)
u_d_normalized = normalize(u_d, u_d_min, u_d_max)

X_d = torch.cat([t_total, x_d_normalized, y_d_normalized], dim=1)
u_exact = u_d_normalized

D_0 = torch.nn.Parameter(torch.ones(1, device=device, requires_grad=True) * np.log(1.3))
r = torch.nn.Parameter(torch.ones(1, device=device, requires_grad=True) * np.log(3))
K = torch.nn.Parameter(torch.ones(1, device=device, requires_grad=True) * np.log(2.6))

optimizer = torch.optim.Adam(list(pinn.parameters()) + [D_0, r, K], lr=1e-3)
D_0s, rs, Ks = [], [], []

D_0_history, r_history, K_history = [], [], []
min_loss = float('inf')
best_params = {}

for i in range(10001):
    optimizer.zero_grad()

    u = pinn(X_d)
    u_t = torch.autograd.grad(u, t_total, torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x_d_normalized, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x_d_normalized, torch.ones_like(u_x), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y_d_normalized, torch.ones_like(u), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y_d_normalized, torch.ones_like(u_y), create_graph=True)[0]

    loss1 = torch.mean((u_t - ((torch.exp(D_0)) * 1000) * (u_xx + u_yy) - ((torch.exp(r)) * 0.1) * u * (1 - u / ((torch.exp(K)) * 1000))) ** 2)
    loss2 = torch.mean((u - u_exact) ** 2)

    loss = (1e-3) * loss1 + loss2
    loss.backward(retain_graph=True)
    optimizer.step()

    D_0_history.append(torch.exp(D_0).item() * 1000)
    r_history.append(torch.exp(r).item() * 0.1)
    K_history.append(torch.exp(K).item() * 1000)

    if loss.item() < min_loss:
        min_loss = loss.item()
        best_params = {
            "D_0": torch.exp(D_0).item() * 1000,
            "r": torch.exp(r).item() * 0.1,
            "K": torch.exp(K).item() * 1000,
        }

    if i % 100 == 0:
        print(f"Step {i}: Loss = {loss.item()}, D_0 = {D_0_history[-1]}, r = {r_history[-1]}, K = {K_history[-1]}")

print("Best parameters at minimum loss:")
print(best_params)